Refactored with love from https://github.com/FUlyankin/matstat-AB/blob/main/week12_likelihood/12_python_mle.ipynb :)

In [78]:
import numpy as np
import pandas as pd
import random
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sc
import math
from scipy import stats 
from scipy.optimize import minimize
import statsmodels.formula.api as smf

In [79]:
random.seed(42)
np.random.seed(42)

$$ Let\:x_1,x_2,\cdots x_n\sim iid\:N(\mu, \sigma^2) $$


$$ then \: f(x)=\frac{1}{\sqrt{2\pi \sigma}}\cdot e^{\frac{(x-\mu)^2}{2\sigma^2}} $$


$$ L(\mu,\sigma^2 \:| \: x_1,\cdots,x_n)=\prod_{i=1}^{n}\frac{1}{\sqrt{2\pi \sigma}}\cdot e^{\frac{(x-\mu)^2}{2\sigma^2}} $$


$$ ln(L) \propto -0.5 \cdot n\cdot ln(\sigma^2)\cdot\sum_{i=1}^{n}\frac{(x_i-\mu)^2}{2\sigma^2} $$

To maximize this function we need to find to find critical points for each parameter and evalute them

$$ \frac{dln(L)}{d\mu}=0, \: \frac{dln(L)}{d\sigma^2}=0 \: \longrightarrow  \:\mu=\frac{\sum_{i=1}^{n}x_i}{n} \: and \:\sigma^2=\frac{\sum_{i=1}^{n}(x_i-\mu)^2}{n} $$

In [80]:
sample = np.random.normal(loc=3, scale=1.7, size=70)

To minimize the function we can use minimize() or fit() if we know the sample distribution

In [81]:
def f(params, x):
    mu = params[0]
    var = params[1]
    n = len(x)

    if var <= 0:
        return np.inf
    l = -0.5 * n * np.log(var) - ( np.sum((x - mu) ** 2) / 2 / var)

    # multiply by -1 because we need to maximize
    return -l

print(f([3, 1.7 ** 2], sample))

65.70002002129968


In [82]:
#Choose values somewhat close to original parameters, var > 0
res = minimize(f, [0, 1], sample)
print(type(res))
print(res)

<class 'scipy.optimize._optimize.OptimizeResult'>
  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 64.23491914414353
        x: [ 2.771e+00  2.305e+00]
      nit: 14
      jac: [-3.815e-06  4.768e-06]
 hess_inv: [[ 3.251e-02  2.141e-03]
            [ 2.141e-03  1.403e-01]]
     nfev: 52
     njev: 17


C:\Users\Alim\AppData\Roaming\Python\Python312\site-packages\scipy\optimize\_numdiff.py:596: RuntimeWarning: invalid value encountered in subtract
  df = fun(x1) - f0


In [83]:
mu, var = res.x
print("Parameters:", mu, var)

Parameters: 2.7709892036843975 2.3054681691579404


In [84]:
mu, var = sc.stats.norm.fit(sample)
print("Parameters:", mu, var ** 2)

Parameters: 2.7709893353074264 2.305467334647894


In [ ]:
print("mu:", np.mean(sample), "var:", np.var(sample)) #Sample mean, variance

mu: 2.7709893353074264 var: 2.3054673346478944


In [86]:
#Covariance matrix
cm = res.hess_inv
#Sample variance
svar = np.var(sample) / len(sample)

alpha = 0.05
z = sc.stats.norm.ppf(1 - alpha / 2)
print("95% CI for mu")
lb = mu - z * np.sqrt(cm[0, 0])
rb = mu + z * np.sqrt(cm[0, 0])

#95% CI
print(f"[{lb}, {rb}] len: {rb - lb}")

95% CI for mu
[2.417606512261488, 3.124372158353365] len: 0.7067656460918768


Freken-Bock sometimes sees ghosts! We have the following data the about number of observed ghosts:

In [87]:
y = [1, 2, 0, 0, 2, 0] #Observed ghosts

Let's assume that the number of observed ghosts follows the Poisson distribution:

$$ P(X=k) = \frac{e^{-\lambda} \lambda^k}{k!}, \: P(x_1, x_2, \cdots x_n \: | \: \lambda) = \prod_{i=1}^{n} \frac{e^{-\lambda}\lambda ^ {x_i}}{x_i!} $$


$$ ln L(x_1,\cdots,x_n \: | \: \lambda) = -n{\lambda} + ln(\lambda)\cdot\sum_{i=1}^{n}x_i - \sum_{i=1}^{n}x_i! \propto -n\lambda + ln(\lambda)\cdot\sum_{i=1}^{n}x_i $$

In [88]:
def f(theta, x) -> float:
    n = len(x)
    if theta <= 0:
        return np.inf
    l = -n * theta + np.log(theta) * np.sum(x)
    return -l

print(y)
res = minimize(f, [1], y)
res

[1, 2, 0, 0, 2, 0]


C:\Users\Alim\AppData\Roaming\Python\Python312\site-packages\scipy\optimize\_numdiff.py:596: RuntimeWarning: invalid value encountered in subtract
  df = fun(x1) - f0


  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 5.911607783969773
        x: [ 8.333e-01]
      nit: 5
      jac: [ 5.960e-08]
 hess_inv: [[ 1.389e-01]]
     nfev: 15
     njev: 7

Now let's assume that the observed number of ghosts related to the amount of congac drunk (in milliliters), amount of cognac follows the normal distribution with some parameters. 

Assume that intensity of Poisson distribution equals to:
$$ \lambda_i = e^{a+b\cdot x_i} $$

Then we need to substitute intensity to our function:
$$ -\sum_{i=1}^{n}\lambda_i + \cdot\sum_{i=1}^{n}y_i\cdot ln(\lambda_i) \: where\: \lambda=e^{a+b\cdot x_i} $$

In [89]:
x =  [3.2, 7.9, 5.4, 4.9, 6.2, 4.3] #Amount of congac
df = pd.DataFrame({'x' : x, 'y' : y})

In [90]:
def f(coef, df) -> float:
    a, b = coef[0], coef[1]
    x, y = np.array(df['x']), np.array(df['y'])
    n = len(x)

    theta = np.exp(a + b * x)

    if np.any(theta <= 0):
        return np.inf
    l = -np.sum(theta) + np.sum( np.log(theta) * y )

    return -l

print(y)
res = minimize(f, [0, 0], df)
res

[1, 2, 0, 0, 2, 0]


  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 4.901374802427498
        x: [-2.599e+00  4.170e-01]
      nit: 9
      jac: [-5.960e-08  0.000e+00]
 hess_inv: [[ 3.642e+00 -5.478e-01]
            [-5.478e-01  8.718e-02]]
     nfev: 39
     njev: 13

This is called a Poisson regression and implemented inside statsmodels package

In [91]:
model = smf.poisson(data=df, formula="y ~ 1 + x")
model.fit().summary()

Optimization terminated successfully.
         Current function value: 1.047945
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                          Poisson Regression Results                          
==============================================================================
Dep. Variable:                      y   No. Observations:                    6
Model:                        Poisson   Df Residuals:                        4
Method:                           MLE   Df Model:                            1
Date:                Thu, 28 May 2026   Pseudo R-squ.:                  0.1384
Time:                        16:58:07   Log-Likelihood:                -6.2877
converged:                       True   LL-Null:                       -7.2979
Covariance Type:            nonrobust   LLR p-value:                    0.1552
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.5988      1.918     -1.355      0.175      -6.358       1.161
x              0.4170      0.297      1.404      0.160      -0.165       0.999
==============================================================================
"""

Properties of MLE:

$$ 1. \: \widehat{\theta}_{mle}\:is\:unbiased $$
$$ 2. \: E(\widehat{\theta}_{mle})\to \theta $$
$$ 3. \: \widehat{\theta}_{mle} \:asymptotically\:efficient \:(CRB \:ineq) $$
$$ 4. \: \widehat{\theta}_{mle} \: asymptotically \: normal\: \widehat{\theta}_{mle}\sim N(\theta, I^{-1}) \: where \: I(\theta)=-E(\frac{d^2lnL(\theta)}{d\theta_i \theta_j})=-E(H) $$
$$5. \: if \: \widehat{\theta}_{mle} \to \theta \: then \: \widehat{\lambda}_{mle} =g(\widehat{\theta}_{mle}) \: (if \: \lambda = g(\theta))$$


$$ Estimate \: is\:efficient\:if \:var(\widehat{\theta}) = \frac{1}{I(\theta)\cdot n} \:(n \ge1) \: where \:I(\theta) =\frac{d^2ln(f)}{d^2\theta}$$